# Employee Attrition Prediction

## Internship Project - Week 2

### Author
Naman Arora

### Objective
Build a Machine Learning model to predict whether an employee is likely to leave the company based on HR analytics data.

# Task 1 - Data Loading & Exploration

In [30]:
import numpy as numpy
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler


In [31]:
df = pd.read_csv("../WA_Fn-UseC_-HR-Employee-Attrition.csv")

print(df.head(10))
print(df.shape)

# checking how many employees have left the company
print(df["Attrition"].value_counts())

# calculating the percentage of employees who have left the company
attrition_percentage = (df["Attrition"].value_counts()["Yes"] / len(df)) * 100
print(f"Percentage of employees who have left the company: {attrition_percentage:.2f}%")


# printing columns that are numeric 
numeric_columns = df.select_dtypes(include=["int64", "float64"]).columns
print(numeric_columns)

# printing columns that are actegorical 
categorical_columns = df.select_dtypes(include=["object"]).columns
print(categorical_columns)

print(f"Total number of numerical columns: {len(numeric_columns)}")
print(f"Total number of categorical columns: {len(categorical_columns)}")

# observation that we made about the attrition rate 
print("Observation: The attrition rate is relatively low, with only a small percentage of employees leaving the company.\nThis suggests that the company may have a stable work environment and good engagement practices that help retain employees.\nHowever, it is important to further analyze the data to identify any potential factors that may contribute to employee attrition and address them proactively.")

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   
5   32        No  Travel_Frequently       1005  Research & Development   
6   59        No      Travel_Rarely       1324  Research & Development   
7   30        No      Travel_Rarely       1358  Research & Development   
8   38        No  Travel_Frequently        216  Research & Development   
9   36        No      Travel_Rarely       1299  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8      

/var/folders/1n/1fwqfjps4v5cmtm5yhyw4kmm0000gn/T/ipykernel_1037/604444428.py:19: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = df.select_dtypes(include=["object"]).columns


### Observation

The dataset is imbalanced because the number of employees who stayed with the company is significantly higher than the number of employees who left. This imbalance should be considered during model training to avoid bias toward the majority class.

# Task 2 - Data Cleaning & Preprocessing

In [32]:
print(df.isnull().sum())

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeCount               0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
Over18                      0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StandardHours               0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSince

### Observation

The dataset contains no missing (null) values in any column. Since all columns have complete data, no missing value treatment such as deletion or imputation is required.

In [33]:
'''
we check constant and identifier columns, standard way   
'''
constant_columns = []

for col in df.columns:
    if df[col].nunique() == 1:
        constant_columns.append(col)

print("Constant Columns:")
print(constant_columns)


identifier_columns = []

for col in df.columns:
    if df[col].nunique() == len(df):
        identifier_columns.append(col)

print("Possible Identifier Columns:")
print(identifier_columns)


# dropping constant and identifier columns
columns_to_drop = constant_columns + identifier_columns

print("Columns to Drop:")
print(columns_to_drop)

df = df.drop(columns=columns_to_drop)

print("Shape before dropping:", (1470, 35))   # or use the shape you noted earlier
print("Shape after dropping:", df.shape)


Constant Columns:
['EmployeeCount', 'Over18', 'StandardHours']
Possible Identifier Columns:
['EmployeeNumber']
Columns to Drop:
['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber']
Shape before dropping: (1470, 35)
Shape after dropping: (1470, 31)


### Observation

The dataset was analyzed to identify columns that do not contribute meaningful information for predicting employee attrition. Constant columns and identifier columns were removed because they provide no useful predictive information and may unnecessarily increase model complexity.

In [34]:
# learnt where to use map practically
df["Attrition"] = df["Attrition"].map({
    "Yes": 1,
    "No": 0
})
print(df["Attrition"].head())

0    1
1    0
2    1
3    0
4    0
Name: Attrition, dtype: int64


### Observation

The target variable was converted from categorical values ("Yes" and "No") to numerical values (1 and 0). This transformation is required because machine learning classification algorithms work with numerical target labels.

In [35]:
columns_to_drop=df.select_dtypes(include=["object"]).columns

df=pd.get_dummies(df, columns=columns_to_drop, drop_first=True)
print(df.head(10))

print(df.shape)

   Age  Attrition  DailyRate  DistanceFromHome  Education  \
0   41          1       1102                 1          2   
1   49          0        279                 8          1   
2   37          1       1373                 2          2   
3   33          0       1392                 3          4   
4   27          0        591                 2          1   
5   32          0       1005                 2          2   
6   59          0       1324                 3          3   
7   30          0       1358                24          1   
8   38          0        216                23          3   
9   36          0       1299                27          3   

   EnvironmentSatisfaction  HourlyRate  JobInvolvement  JobLevel  \
0                        2          94               3         2   
1                        3          61               2         2   
2                        4          92               2         1   
3                        4          56               3  

/var/folders/1n/1fwqfjps4v5cmtm5yhyw4kmm0000gn/T/ipykernel_1037/3941537351.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  columns_to_drop=df.select_dtypes(include=["object"]).columns


### Observation

All categorical features were converted into numerical features using One-Hot Encoding. This transformation enables machine learning algorithms to process categorical information without introducing artificial ordering between categories. The `drop_first=True` parameter was used to avoid redundant features.

In [38]:
# axis=1 tells that we need to think horizontally and it is a value in a column not the column name itself
df1=df.drop("Attrition",axis=1) 
df2=df["Attrition"]

numeric_columns=df1.select_dtypes(include=["int64", "float64"]).columns

df1[numeric_columns]=StandardScaler().fit_transform(df1[numeric_columns])

print(df1.head(10))



        Age  DailyRate  DistanceFromHome  Education  EnvironmentSatisfaction  \
0  0.446350   0.742527         -1.010909  -0.891688                -0.660531   
1  1.322365  -1.297775         -0.147150  -1.868426                 0.254625   
2  0.008343   1.414363         -0.887515  -0.891688                 1.169781   
3 -0.429664   1.461466         -0.764121   1.061787                 1.169781   
4 -1.086676  -0.524295         -0.887515  -1.868426                -1.575686   
5 -0.539166   0.502054         -0.887515  -0.891688                 1.169781   
6  2.417384   1.292887         -0.764121   0.085049                 0.254625   
7 -0.758170   1.377177          1.827158  -1.868426                 1.169781   
8  0.117845  -1.453958          1.703764   0.085049                 1.169781   
9 -0.101159   1.230910          2.197341   0.085049                 0.254625   

   HourlyRate  JobInvolvement  JobLevel  JobSatisfaction  MonthlyIncome  ...  \
0    1.383138        0.379672 -0.057788

### Observation

The continuous numerical features were standardized using StandardScaler. Standardization ensures that features with larger numerical ranges do not dominate the learning process of machine learning algorithms. The target variable was kept separate and was not scaled.